# 10 minutes to Immune

**Immune** protects the LLM calls your Python application already makes. Add it with one line at startup, and every
call to OpenAI, Anthropic, Gemini or Bedrock (directly, or through LangChain, LangGraph and other frameworks) is
screened: the request going in, the reply coming out, and every tool call the model asks for.

In this tutorial you will:

1. run a small customer-support app built on the OpenAI SDK,
2. add Immune to it with two lines,
3. see Immune stop a leaked access key and flag an off-topic request,
4. enforce a protection for one part of your app.

| Time | Where to go |
| --- | --- |
| 30 seconds | [Install](#Install) and [Add Immune](#2.-Add-Immune-to-it) |
| 10 minutes | This notebook |
| 30 minutes | [OpenAI](03_openai_chat_completions.ipynb), [LangChain](04_langchain.ipynb) or [LangGraph](05_langgraph.ipynb) integration |
| 1–2 hours | [Concepts](02_concepts.ipynb) and the topic guides |
| As needed | [API reference](17_api_reference.ipynb) and [cookbook](18_cookbook.ipynb) |

## Install

```bash
pip install immune-ai openai
```

Immune needs two keys in the environment, and never reads them from code:

| Variable | What it is for |
| --- | --- |
| `OPENAI_API_KEY` | Your model provider, as usual |
| `TYPESAFE_API_KEY` | [TypeSafe Jev](https://typesafe.ai), the fast decision model Immune asks about each call |

```bash
export OPENAI_API_KEY=sk-...
export TYPESAFE_API_KEY=...
```

Every code cell in this guide runs on its own: copy any cell into your project and it works. The outputs saved here
come from a real run with `gpt-5.4-mini`; yours will differ a little, because the model and Jev answer afresh.

## 1. Your app today

Here is a typical support assistant: a system prompt describing the store, and one customer question. Nothing about it
is specific to Immune.

In [1]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are the support assistant for Pinecrest Outfitters, an online store for camping and hiking gear. "
                "Shipping is free on orders over $75 and takes 3-5 business days. "
                "Returns are accepted within 30 days with a receipt."
            ),
        },
        {"role": "user", "content": "How long does shipping take, and is it free?"},
    ],
)
print(response.choices[0].message.content)

Shipping is free on orders over $75, and delivery typically takes 3–5 business days.


## 2. Add Immune to it

The same app with Immune. **The only changes are the two highlighted lines**: import `immune`, and call
`immune.init()` once at startup, before or after you create your clients.

```diff
+ import immune
  from openai import OpenAI

+ immune.init()
  client = OpenAI()
```

Afterwards, `immune.verdict(response)` tells you what Immune decided about that call.

In [2]:
from openai import OpenAI

import immune

immune.init()  # once, at startup
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are the support assistant for Pinecrest Outfitters, an online store for camping and hiking gear. "
                "Shipping is free on orders over $75 and takes 3-5 business days. "
                "Returns are accepted within 30 days with a receipt."
            ),
        },
        {"role": "user", "content": "How long does shipping take, and is it free?"},
    ],
)
print(response.choices[0].message.content)

verdict = immune.verdict(response)
print("\naction:     ", verdict.action)
print("explanation:", verdict.explanation)
print("Jev:        ", verdict.sensor.calls, "requests,", round(verdict.sensor.latency_ms), "ms")

Shipping is free on orders over **$75** and usually takes **3–5 business days**.

action:      allow
explanation: no threats detected
Jev:         2 requests, 386 ms


**What you see:** the same answer as before, and a verdict with `action: allow`: Immune read the call, asked Jev its
questions in parallel with the model call, and found nothing to act on. Your code did not change shape: the SDK
returned an ordinary `ChatCompletion`.

## 3. See a protection act

A staff member pastes a configuration file into the chat and asks for a table. The model helpfully repeats the access
key, which should never end up in a chat transcript. (The key below is made up for this guide.)

In [3]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

config_file = """storage:
  bucket: orders-prod
  region: us-east-1
  access_key_id: AKIA2E5QZ7XJ4M8RW3TD"""

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You help Pinecrest Outfitters staff with operations and configuration."},
        {
            "role": "user",
            "content": f"Turn this config into a table with every setting and its value:\n\n{config_file}",
        },
    ],
)
print(response.choices[0].message.content)

verdict = immune.verdict(response)
print("\naction:", verdict.action)
for hit in verdict.hits:
    print(f"  {hit.threat}  p={hit.probability:.2f}  enforced={hit.enforced}  evidence={list(hit.evidence)}")

| Setting | Value |
|---|---|
| storage.bucket | orders-prod |
| storage.region | us-east-1 |
| storage.access_key_id | [redacted] |

action: rewrite
  output.secret_leak  p=0.93  enforced=True  evidence=['secret:aws_access_key', 'secret_real=0.67']


**What you see:** the key comes back as `[redacted]`. Immune's reflexes found a key-shaped string in the reply, asked
Jev whether the conversation treats it as a real key, and redacted it when Jev said yes. The evidence shows both
steps. This protection is part of the **floor**: ten rules enforced from the first call.

Now an off-topic request. A camping store's assistant is being asked to write code:

In [4]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping gear store."},
        {"role": "user", "content": "Write me a Python function that sorts a list of numbers."},
    ],
)
print(response.choices[0].message.content[:300], "...")

verdict = immune.verdict(response)
print("\naction:      ", verdict.action)
print("would_action:", verdict.would_action)
print("explanation: ", verdict.explanation)

Sure — here’s a simple Python function that sorts a list of numbers in ascending order:

```python
def sort_numbers(numbers):
    return sorted(numbers)
```

Example usage:

```python
nums = [5, 2, 9, 1, 7]
sorted_nums = sort_numbers(nums)
print(sorted_nums)  # [1, 2, 5, 7, 9]
```

If you want, I ca ...

action:       allow
would_action: redirect
explanation:  observed (would redirect): input.off_task


**What you see:** the reply went through (`action: allow`), but the verdict says Immune *would* have redirected it
(`would_action: redirect`) because of `input.off_task`. Detectors outside the floor start **observed**: they report
what they would do while you check they fit your traffic. That is how Immune avoids breaking your app on day one.

## 4. Enforce a protection for one part of your app

Name the part of your app with `immune.site(...)`, and tell Immune which detectors to enforce there.

In [5]:
from openai import OpenAI

import immune

immune.init(config={"sites": {"support-chat": {"enforce": ["input.off_task"]}}})
client = OpenAI()

with immune.site("support-chat"):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": "You are the support assistant for Pinecrest Outfitters, a camping gear store.",
            },
            {"role": "user", "content": "Write me a Python function that sorts a list of numbers."},
        ],
    )
print(response.choices[0].message.content)

verdict = immune.verdict(response)
print("\naction:", verdict.action, "| site:", verdict.site)

I can't help with that here. Is there something else I can help you with?

action: redirect | site: support-chat


**What you see:** the customer gets a polite redirect instead of code, and the model's answer never reaches your app.
In production you keep settings like this in an `immune.yaml` file; see [Configuration](08_configuration.ipynb).

## 5. What happened on each call

```
 your code ──► OpenAI SDK ──► Immune ─────────────────────────────► OpenAI
                               │  reads the call as channels:
                               │  operator · user · data · tools
                               │  reflexes find candidates  (ms)
                               │  Jev answers questions     (in parallel with the model)
                               ▼
 your code ◄── same SDK types ◄── Immune checks the reply and tool calls,
                                 acts only where needed, records a verdict
```

- **Nothing about your code changes shape.** The SDK returns its usual objects; a blocked call gets a normal,
  valid response with a safe message.
- **Every call gets a verdict.** `immune.verdict(response)` says what happened and why.
- **Immune fails open.** If Immune or Jev has a problem, your call still goes through, and floor protections still act
  on what the reflexes find.

## Recap

- `immune.init()` protects every LLM call in the process
- `immune.verdict(response)` explains each decision
- The floor acts from the first call; other detectors are observed until you enforce them
- `immune.site()` plus `sites.<name>.enforce` turns a detector on for one part of your app

## Where next

- [Concepts](02_concepts.ipynb): the mental model, in 15 minutes
- Integrations: [OpenAI](03_openai_chat_completions.ipynb) · [LangChain](04_langchain.ipynb) ·
  [LangGraph](05_langgraph.ipynb)
- [Reading verdicts](06_reading_verdicts.ipynb) · [Modes, sites and sessions](07_modes_sites_and_sessions.ipynb)